<a href="https://colab.research.google.com/github/NoorDataAnalyst/flyrank-ML-Internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


The output shows the top 10 pages that should be considered for a content refresh based on the rule created. These pages have not been updated for at least 180 days and are still receiving a high number of impressions, meaning they continue to appear frequently in Google search results. They are ranked by the number of impressions, so the pages with the highest visibility appear first. Although the rule did not use the trend_direction column, all of the top pages also have a "down" trend, indicating that their performance is declining over time. This suggests that updating these high-visibility but outdated pages could help improve their search performance and potentially increase traffic.

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


The hand-crafted rule performs quite well for the highest-ranked pages, but its accuracy decreases as more pages are included. A Precision@20 of 0.900 means that 18 out of the top 20 pages recommended by the rule are actually declining, indicating that the rule is effective at identifying the most important pages to refresh. However, the Precision@50 drops to 0.680, meaning that 34 out of the top 50 pages are actually declining while 16 are not. This suggests that the rule is very reliable for the top recommendations but becomes less accurate as the ranking extends further. In other words, the highest-ranked pages are strong candidates for content refresh, while lower-ranked pages are more likely to include false positives.

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



The decision tree automatically learned a simple set of rules to predict whether a page is declining. It first checks whether the page receives any meaningful number of impressions. Pages with almost no impressions are predicted as not declining because they have little search visibility. For pages that do receive impressions, the model then considers the content's age. Pages with content that is approximately 313 days old or newer are predicted as declining, while pages with older content are predicted as not declining. Because the tree is limited to a depth of two, the learned rules are easy to read and interpret, making it a transparent machine learning model compared with more complex algorithms.

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


In this experiment, the hand-crafted rule outperformed the depth-2 decision tree at both Precision@20 and Precision@50. This indicates that the simple expert-defined rule was more effective at identifying declining pages among the highest-ranked recommendations. The decision tree's performance was limited by its shallow depth, which restricted it to learning only a few simple decision rules. These results suggest that, for this dataset, the manually designed ranking rule provides more accurate recommendations than the simple decision tree model.

Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The decision tree achieved a perfect Precision@50 of 1.000 because it was given trend_pct, a feature that directly determines whether a page is labeled as declining. Since the target label (is_declining_label) is derived from trend_direction, which in turn is based on trend_pct, the model is effectively using the answer to predict the answer. This is known as data leakage. Although the model appears to perform perfectly, its performance is misleading because it would not have access to trend_pct when making predictions on new, unseen pages. In real-world machine learning, only information available before the outcome occurs should be used as features, otherwise the evaluation will be unrealistically optimistic and the model will not generalize well.

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

### **Experiment 1: Increase tree depth to 3**

In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Features
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Prepare data
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# Train tree
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree_depth3.fit(X, y)

# Print tree
print(export_text(tree_depth3, feature_names=features))

# Evaluate
tree_score3 = tree_depth3.predict_proba(X)[:, 1]

for k in (20, 50):
    print(f"Depth 3 Precision@{k}: {precision_at_k(tree_score3, y, k):.3f}")

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

Depth 3 Precision@20: 0.700
Depth 3 Precision@50: 0.720


### **Experiment 2: Increase tree depth to 4**

In [8]:
# Your experiment here
from sklearn.tree import DecisionTreeClassifier, export_text

tree_depth4 = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree_depth4.fit(X, y)

print(export_text(tree_depth4, feature_names=features))

tree_score4 = tree_depth4.predict_proba(X)[:, 1]

for k in (20, 50):
    print(f"Depth 4 Precision@{k}: {precision_at_k(tree_score4, y, k):.3f}")

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- impressions_90d <= 73.50
|   |   |   |   |--- class: 1
|   |   |  

### **Experiment 3: Remove impressions_90d**

In [9]:
features_no_impressions = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count"
]

X_no_imp = (
    df[features_no_impressions]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

tree_no_imp = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_no_imp.fit(X_no_imp, y)

print(export_text(tree_no_imp, feature_names=features_no_impressions))

tree_score_no_imp = tree_no_imp.predict_proba(X_no_imp)[:,1]

for k in (20,50):
    print(f"No Impressions Precision@{k}: {precision_at_k(tree_score_no_imp, y, k):.3f}")

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

No Impressions Precision@20: 0.850
No Impressions Precision@50: 0.700


### **Experiment 4: Add engagement_rate (if it exists)**

In [10]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'hand_rule_score']


In [11]:
features_engagement = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_eng = (
    df[features_engagement]
    .replace([np.inf,-np.inf],np.nan)
    .fillna(0)
)

tree_eng = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_eng.fit(X_eng,y)

print(export_text(tree_eng, feature_names=features_engagement))

tree_score_eng = tree_eng.predict_proba(X_eng)[:,1]

for k in (20,50):
    print(f"Engagement Precision@{k}: {precision_at_k(tree_score_eng,y,k):.3f}")

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

Engagement Precision@20: 0.850
Engagement Precision@50: 0.700


### **Experiment 5: Proper Train/Test Split**

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

X = (
    df[features]
    .replace([np.inf,-np.inf],np.nan)
    .fillna(0)
)

y = df["is_declining_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

print(export_text(tree, feature_names=features))

test_scores = tree.predict_proba(X_test)[:,1]

for k in (20,50):
    print(f"Test Precision@{k}: {precision_at_k(test_scores,y_test,k):.3f}")

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.65
|   |   |--- class: 0
|   |--- avg_position >  0.65
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 344.50
|   |   |--- class: 1
|   |--- content_age_days >  344.50
|   |   |--- class: 0

Test Precision@20: 0.650
Test Precision@50: 0.620


### **Experiment 6: Compare Depth 2 vs 3 vs 4**

In [13]:
depths = [2,3,4]

for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X,y)

    scores = model.predict_proba(X)[:,1]

    print("="*40)
    print(f"Tree Depth = {depth}")

    for k in (20,50):
        print(f"Precision@{k}: {precision_at_k(scores,y,k):.3f}")

Tree Depth = 2
Precision@20: 0.550
Precision@50: 0.600
Tree Depth = 3
Precision@20: 0.700
Precision@50: 0.720
Tree Depth = 4
Precision@20: 0.600
Precision@50: 0.680


**Experiment Results**

I conducted several experiments to evaluate how changes to the decision tree affected its performance and interpretability.

First, I increased the maximum tree depth from 2 to 3 and then to 4. With depth = 2, the model achieved a Precision@20 of 0.550 and Precision@50 of 0.600. Increasing the depth to 3 improved the results to 0.700 and 0.720, respectively, making it the best-performing tree. However, increasing the depth further to 4 reduced the performance to 0.600 at Precision@20 and 0.680 at Precision@50. This suggests that a depth of 3 captured more useful patterns, while a deeper tree did not provide additional benefit and became more complex to interpret.

Next, I removed the impressions_90d feature. Without this feature, the tree selected avg_position as its first split instead of impressions_90d. The model achieved a Precision@20 of 0.850 and Precision@50 of 0.700, indicating that avg_position was also a strong predictor of declining pages.

I then added engagement_rate while keeping impressions_90d removed. The tree still chose avg_position as the first split, and the performance remained unchanged (Precision@20 = 0.850, Precision@50 = 0.700). This suggests that, for this dataset, engagement_rate did not provide additional predictive value beyond the existing features.

Finally, I evaluated the model using a train/test split instead of scoring on the same data used for training. The test results were Precision@20 = 0.650 and Precision@50 = 0.620, which are lower than the in-sample scores. This is expected because the model was evaluated on unseen data, providing a more realistic estimate of its performance. The decrease in precision indicates that the in-sample evaluation was somewhat optimistic.

Overall, the experiments show that a decision tree with a maximum depth of 3 provided the best balance between predictive performance and interpretability. While increasing the depth beyond 3 made the tree more complex, it did not improve performance. The experiments also showed that avg_position became the most important feature when impressions_90d was removed, and that using a proper train/test split produced more reliable performance estimates than evaluating on the training data itself.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.